In [1]:
import numpy as np
import pandas as pd
from scipy import stats

In [2]:
active_validators_size = pd.read_csv('../int/active_validators_size_change.csv')
active_validators_category = pd.read_csv('../int/active_validators_category_change.csv')
active_validators_pool = pd.read_csv('../int/active_validators_pool_change.csv')

In [3]:
active_validators_pool = active_validators_pool.drop(columns=('Unnamed: 0'))
active_validators_category = active_validators_category.drop(columns=('Unnamed: 0'))
active_validators_size = active_validators_size.drop(columns=('Unnamed: 0'))

In [4]:
aave = pd.read_csv('../int/aave_grouped.csv', usecols=('slot', 'liquidity_apr'))
aave['price_pct_change'] = aave['liquidity_apr'].pct_change()
aave = aave.dropna()
aave = aave[aave['slot'].isin(active_validators_size['slot'])]
aave = aave[aave['slot'] >= 6206400]
aave

/var/folders/mb/5hm6pgrs3zj_1m_kgvpt40jw0000gn/T/ipykernel_24049/2329489729.py:2: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  aave['price_pct_change'] = aave['liquidity_apr'].pct_change()


,slot,liquidity_apr,price_pct_change
75,6206400.0,2.301241,0.010450
76,6213600.0,2.441569,0.060979
77,6220800.0,2.242568,-0.081506
78,6228000.0,2.073961,-0.075185
79,6235200.0,1.928353,-0.070208
...,...,...,...
457,8956800.0,1.371801,-0.037020
458,8964000.0,1.377068,0.003839
459,8971200.0,1.370012,-0.005124
460,8978400.0,1.308489,-0.044907


In [5]:

# Merge the two DataFrames on the slot column
price_elasticity_size = pd.merge(active_validators_size, aave[['slot', 'price_pct_change']], on='slot')

# Drop rows with infinite values
price_elasticity_size.replace([np.inf, -np.inf], np.nan, inplace=True)

# Calculate elasticity for 'total' first
price_elasticity_size['elasticity_total'] = price_elasticity_size['total'] / price_elasticity_size['price_pct_change']
price_elasticity_size['elasticity_total'] = price_elasticity_size['elasticity_total'].replace([np.inf, -np.inf], np.nan)

# Initialize dictionaries to store elasticity values, standard deviations, t-statistics, p-values, and number of valid rows (N)
elasticity = {}
standard_deviations = {}
t_statistics = {}
p_values = {}
valid_counts = {}

# Calculate elasticity for the 'total' column
valid_total_elasticity = price_elasticity_size['elasticity_total'].dropna()
elasticity['total'] = valid_total_elasticity.mean()
standard_deviations['total'] = valid_total_elasticity.std()
t_statistics['total'] = '-'
p_values['total'] = '-'
valid_counts['total'] = len(valid_total_elasticity)

# Calculate elasticity for each column except 'total'
columns = ['1', '2-5', '6-19', '20-99', '100+']
for col in columns:
    price_elasticity_size[f'elasticity_{col}'] = price_elasticity_size[col] / price_elasticity_size['price_pct_change']
    
    # Replace infinite values with NaN
    price_elasticity_size[f'elasticity_{col}'] = price_elasticity_size[f'elasticity_{col}'].replace([np.inf, -np.inf], np.nan)
    
    # Drop NaN values for calculation purposes
    valid_elasticity = price_elasticity_size[f'elasticity_{col}'].dropna()
    
    elasticity[col] = valid_elasticity.mean()
    standard_deviations[col] = valid_elasticity.std()
    
    # Perform a two-sample t-test against the 'total' elasticity
    t_stat, p_value = stats.ttest_ind(valid_elasticity, valid_total_elasticity, equal_var=False)
    t_statistics[col] = t_stat
    p_values[col] = p_value
    
    # Store the number of valid rows
    valid_counts[col] = len(valid_elasticity)

# Print the results in the requested format
print("Elasticity Analysis Results:")
print(f'total: Mean Elasticity = {elasticity["total"]}, t(Mean) = {t_statistics["total"]}, SD = {standard_deviations["total"]}, N = {valid_counts["total"]}, p-value = {p_values["total"]}')
for col in columns:
    print(f'{col}: Mean Elasticity = {elasticity[col]}, t(Mean) = {t_statistics[col]}, SD = {standard_deviations[col]}, N = {valid_counts[col]}, p-value = {p_values[col]}')

# Display the DataFrame with elasticity columns
print(price_elasticity_size)

Elasticity Analysis Results:
total: Mean Elasticity = -0.04733169314494815, t(Mean) = -, SD = 4.002720841279244, N = 387, p-value = -
1: Mean Elasticity = 0.2536952890083262, t(Mean) = 1.091681303257403, SD = 3.661167465106686, N = 387, p-value = 0.27531647782805224
2-5: Mean Elasticity = -0.4216546962052726, t(Mean) = -0.7868383039564637, SD = 8.459544113623448, N = 387, p-value = 0.4317150712346022
6-19: Mean Elasticity = -0.5131701887749606, t(Mean) = -0.9781885461165236, SD = 8.470318752583335, N = 387, p-value = 0.32841097460413526
20-99: Mean Elasticity = 0.12524946627211092, t(Mean) = 0.5208044736841159, SD = 5.145313566391897, N = 387, p-value = 0.6026613593922203
100+: Mean Elasticity = -0.04267160360527805, t(Mean) = 0.015368252928306647, SD = 4.42288089977937, N = 387, p-value = 0.9877424009316235
          slot         1      100+       2-5     20-99      6-19     total  \
0    6206400.0  0.015042 -0.015342  0.000000  0.018732  0.000000 -0.012792   
1    6213600.0  0.000000

In [6]:

# Merge the two DataFrames on the slot column
price_elasticity_category = pd.merge(active_validators_category, aave[['slot', 'price_pct_change']], on='slot')

# Drop rows with infinite values
price_elasticity_category.replace([np.inf, -np.inf], np.nan, inplace=True)

# Calculate elasticity for 'total' first
price_elasticity_category['elasticity_total'] = price_elasticity_category['total'] / price_elasticity_category['price_pct_change']
price_elasticity_category['elasticity_total'] = price_elasticity_category['elasticity_total'].replace([np.inf, -np.inf], np.nan)

# Initialize dictionaries to store elasticity values, standard deviations, t-statistics, p-values, and number of valid rows (N)
elasticity = {}
standard_deviations = {}
t_statistics = {}
p_values = {}
valid_counts = {}

# Calculate elasticity for the 'total' column
valid_total_elasticity = price_elasticity_category['elasticity_total'].dropna()
elasticity['total'] = valid_total_elasticity.mean()
standard_deviations['total'] = valid_total_elasticity.std()
t_statistics['total'] = '-'
p_values['total'] = '-'
valid_counts['total'] = len(valid_total_elasticity)

# Calculate elasticity for each column except 'total'
columns = active_validators_category.columns.drop('slot')
for col in columns:
    price_elasticity_category[f'elasticity_{col}'] = price_elasticity_category[col] / price_elasticity_category['price_pct_change']
    
    # Replace infinite values with NaN
    price_elasticity_category[f'elasticity_{col}'] = price_elasticity_category[f'elasticity_{col}'].replace([np.inf, -np.inf], np.nan)
    
    # Drop NaN values for calculation purposes
    valid_elasticity = price_elasticity_category[f'elasticity_{col}'].dropna()
    
    elasticity[col] = valid_elasticity.mean()
    standard_deviations[col] = valid_elasticity.std()
    
    # Perform a two-sample t-test against the 'total' elasticity
    t_stat, p_value = stats.ttest_ind(valid_elasticity, valid_total_elasticity, equal_var=False)
    t_statistics[col] = t_stat
    p_values[col] = p_value
    
    # Store the number of valid rows
    valid_counts[col] = len(valid_elasticity)

# Print the results in the requested format
print("Elasticity Analysis Results:")
print(f'total: Mean Elasticity = {elasticity["total"]}, t(Mean) = {t_statistics["total"]}, SD = {standard_deviations["total"]}, N = {valid_counts["total"]}, p-value = {p_values["total"]}')
for col in columns:
    print(f'{col}: Mean Elasticity = {elasticity[col]}, t(Mean) = {t_statistics[col]}, SD = {standard_deviations[col]}, N = {valid_counts[col]}, p-value = {p_values[col]}')

# Display the DataFrame with elasticity columns
print(price_elasticity_category)

Elasticity Analysis Results:
total: Mean Elasticity = -0.04733169314494815, t(Mean) = 0.0, SD = 4.002720841279244, N = 387, p-value = 1.0
CEX: Mean Elasticity = 0.7360007188119754, t(Mean) = 0.7292073849397297, SD = 20.749940929751236, N = 387, p-value = 0.4662864611152703
Liquid Restaking: Mean Elasticity = 0.26123748619615794, t(Mean) = 0.21768219152372348, SD = 27.59716102364224, N = 387, p-value = 0.8277871229529418
Liquid Staking: Mean Elasticity = 0.15469034302937187, t(Mean) = 0.4461092277185903, SD = 7.958811540533165, N = 387, p-value = 0.6556878996744743
Solo Stakers: Mean Elasticity = 0.21212545923874346, t(Mean) = 0.6015051278715946, SD = 7.482204298590869, N = 387, p-value = 0.5477345570894888
Staking Pools: Mean Elasticity = -1.0074782953535435, t(Mean) = -0.7567249945222222, SD = 24.637564491086636, N = 387, p-value = 0.449653138525667
Unidentified: Mean Elasticity = -0.7844644286099964, t(Mean) = -1.029699024414863, SD = 13.502043835512076, N = 387, p-value = 0.30370037

In [7]:

# Merge the two DataFrames on the slot column
price_elasticity_pool = pd.merge(active_validators_pool, aave[['slot', 'price_pct_change']], on='slot')

# Drop rows with infinite values
price_elasticity_pool.replace([np.inf, -np.inf], np.nan, inplace=True)

# Calculate elasticity for 'total' first
price_elasticity_pool['elasticity_total'] = price_elasticity_pool['total'] / price_elasticity_pool['price_pct_change']
price_elasticity_pool['elasticity_total'] = price_elasticity_pool['elasticity_total'].replace([np.inf, -np.inf], np.nan)

# Initialize dictionaries to store elasticity values, standard deviations, t-statistics, p-values, and number of valid rows (N)
elasticity = {}
standard_deviations = {}
t_statistics = {}
p_values = {}
valid_counts = {}

# Calculate elasticity for the 'total' column
valid_total_elasticity = price_elasticity_pool['elasticity_total'].dropna()
elasticity['total'] = valid_total_elasticity.mean()
standard_deviations['total'] = valid_total_elasticity.std()
t_statistics['total'] = '-'
p_values['total'] = '-'
valid_counts['total'] = len(valid_total_elasticity)

# Calculate elasticity for each column except 'total'
columns = ['Lido', 'Coinbase', 'Binance', 'Rocketpool', 'Kraken', 'OKX', 'Bitcoin Suisse', 'Ledger Live', 'Ether.Fi', 'Mantle', 'Other Stakers']
for col in columns:
    price_elasticity_pool[f'elasticity_{col}'] = price_elasticity_pool[col] / price_elasticity_pool['price_pct_change']
    
    # Replace infinite values with NaN
    price_elasticity_pool[f'elasticity_{col}'] = price_elasticity_pool[f'elasticity_{col}'].replace([np.inf, -np.inf], np.nan)
    
    # Drop NaN values for calculation purposes
    valid_elasticity = price_elasticity_pool[f'elasticity_{col}'].dropna()
    
    elasticity[col] = valid_elasticity.mean()
    standard_deviations[col] = valid_elasticity.std()
    
    # Perform a two-sample t-test against the 'total' elasticity
    t_stat, p_value = stats.ttest_ind(valid_elasticity, valid_total_elasticity, equal_var=False)
    t_statistics[col] = t_stat
    p_values[col] = p_value
    
    # Store the number of valid rows
    valid_counts[col] = len(valid_elasticity)

# Print the results in the requested format
print("Elasticity Analysis Results:")
print(f'total: Mean Elasticity = {elasticity["total"]}, t(Mean) = {t_statistics["total"]}, SD = {standard_deviations["total"]}, N = {valid_counts["total"]}, p-value = {p_values["total"]}')
for col in columns:
    print(f'{col}: Mean Elasticity = {elasticity[col]}, t(Mean) = {t_statistics[col]}, SD = {standard_deviations[col]}, N = {valid_counts[col]}, p-value = {p_values[col]}')

# Display the DataFrame with elasticity columns
print(price_elasticity_pool)

Elasticity Analysis Results:
total: Mean Elasticity = -0.04733169314494815, t(Mean) = -, SD = 4.002720841279244, N = 387, p-value = -
Lido: Mean Elasticity = 0.1771197655610607, t(Mean) = 0.4748780402228148, SD = 8.392468797112619, N = 387, p-value = 0.6350613993494363
Coinbase: Mean Elasticity = -0.3808161300626598, t(Mean) = -0.5843565325622574, SD = 10.488928357140525, N = 387, p-value = 0.559246097497494
Binance: Mean Elasticity = 6.700919397042662, t(Mean) = 1.0171144089408954, SD = 130.4585619967319, N = 387, p-value = 0.3097348941486013
Rocketpool: Mean Elasticity = -0.1068339121772701, t(Mean) = -0.16927466935986166, SD = 5.638833102993071, N = 387, p-value = 0.8656297677256723
Kraken: Mean Elasticity = 0.41304979972604344, t(Mean) = 0.4183731723558279, SD = 21.274310482939022, N = 387, p-value = 0.6758916056735098
OKX: Mean Elasticity = 3.4798606127707514, t(Mean) = 1.2466497123420153, SD = 55.515499716872974, N = 387, p-value = 0.21327415975432806
Bitcoin Suisse: Mean Elastic